# Detector de estado de frutas y verduras con YOLO entrenado con imagenes de COCO
Esta libreta hace todo el flujo del proyecto:
1. Descarga y prepara el dataset binario Healthy vs Rotten.
2. Entrena un clasificador YOLO11-cls.
3. Evalua en val y test.
4. Ejecuta una prueba automatica con una imagen.
5. Visualiza metricas con matplotlib.
6. Ejecuta inferencia sobre una imagen o una carpeta.

In [1]:
import os
from pathlib import Path

# --- 1. CONFIGURACIÓN DE AMD Y CARPETAS TEMPORALES ---
os.environ["HSA_OVERRIDE_GFX_VERSION"] = "9.0.10" 
os.environ["ROCM_PATH"] = "/opt/rocm"

# Creamos la carpeta temporal en tu espacio
carpeta_tmp = Path.cwd() / ".tmp_miopen"
carpeta_tmp.mkdir(exist_ok=True)

# ¡El truco maestro! Engañamos a Linux y a Boost para que usen tu carpeta
os.environ["TMPDIR"] = str(carpeta_tmp)
os.environ["TEMP"] = str(carpeta_tmp)
os.environ["TMP"] = str(carpeta_tmp)

# Y por si acaso, se lo repetimos a MIOpen
os.environ["MIOPEN_USER_DB_PATH"] = str(carpeta_tmp)
os.environ["MIOPEN_CUSTOM_CACHE_DIR"] = str(carpeta_tmp)

In [2]:
import random
import shutil
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
import cv2 
import matplotlib.pyplot as plt
import pandas as pd
import torch
import mlflow
from tqdm.auto import tqdm
from ultralytics import YOLO
import kagglehub

DATASET_SLUG = "muhammad0subhan/fruit-and-vegetable-disease-healthy-vs-rotten"
DATA_DIR = Path.cwd() / "fruit_binary_yolo_cls"
BASE_MODEL = "yolo11n-cls.pt"
TRAIN_RUN_NAME = "fruit_hs_vs_rt_cls_simple_ft5"
DETECTOR_WEIGHTS = "yolo11n.pt"
CLASSIFIER_WEIGHTS = Path("runs/classify") / TRAIN_RUN_NAME / "weights" / "best.pt"
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
PRODUCE_CLASS_NAMES = {"apple", "banana", "orange", "broccoli", "carrot"}


if torch.cuda.is_available():
    # En sistemas AMD con ROCm, esto devolverá True
    DEVICE = 0 
    print(f"Usando GPU AMD/NVIDIA: {torch.cuda.get_device_name(0)}")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    # Por si alguna vez programas en una Mac con chip M1/M2/M3
    DEVICE = "mps"
    print("Usando Apple Metal (MPS)")
else:
    DEVICE = "cpu"
    print("Usando CPU (No se detectó GPU compatible)")

print("Data dir:", DATA_DIR)
print("Base model:", BASE_MODEL)
print("Run name:", TRAIN_RUN_NAME)

Usando GPU AMD/NVIDIA: AMD Instinct MI210
Data dir: /lustre/cursos/curso04/estudiante_66/computerVision/fruit_binary_yolo_cls
Base model: yolo11n-cls.pt
Run name: fruit_hs_vs_rt_cls_simple_ft5


In [3]:
def find_dataset_root(download_path):
    candidates = [download_path] + [path for path in download_path.iterdir() if path.is_dir()]
    for candidate in candidates:
        subdirs = [path.name for path in candidate.iterdir() if path.is_dir()]
        has_healthy = any(name.endswith("_Healthy") for name in subdirs)
        has_rotten = any(name.endswith("_Rotten") for name in subdirs)
        if has_healthy and has_rotten:
            return candidate
    raise FileNotFoundError("No se encontro una carpeta con estructura *_Healthy / *_Rotten")

def collect_images(dataset_root):
    grouped_paths = defaultdict(list)
    for class_dir in sorted(path for path in dataset_root.iterdir() if path.is_dir()):
        name = class_dir.name
        if name.endswith("_Healthy"):
            label = "Healthy"
        elif name.endswith("_Rotten"):
            label = "Rotten"
        else:
            continue

        files = [path for path in class_dir.rglob("*") if path.is_file() and path.suffix.lower() in IMAGE_EXTS]
        grouped_paths[label].extend(files)
    return grouped_paths

def stratified_split(items_by_label, train_ratio=0.8, val_ratio=0.1, test_ratio=0.1, seed=42):
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-9
    rng = random.Random(seed)

    split = {
        "train": defaultdict(list),
        "val": defaultdict(list),
        "test": defaultdict(list),
    }

    for label, items in items_by_label.items():
        items = items.copy()
        rng.shuffle(items)

        count = len(items)
        count_train = int(count * train_ratio)
        count_val = int(count * val_ratio)

        split["train"][label] = items[:count_train]
        split["val"][label] = items[count_train:count_train + count_val]
        split["test"][label] = items[count_train + count_val:]

    return split

def count_images(root):
    summary = {}
    for split_name in ["train", "val", "test"]:
        summary[split_name] = {}
        for label in ["Healthy", "Rotten"]:
            label_dir = root / split_name / label
            if label_dir.exists():
                count = len([path for path in label_dir.rglob("*") if path.is_file() and path.suffix.lower() in IMAGE_EXTS])
                summary[split_name][label] = count
            else:
                summary[split_name][label] = 0
    return summary

def link_or_copy(src, dst):
    try:
        os.link(src, dst)
    except OSError:
        shutil.copy2(src, dst)

# 5. Función Principal de Preparación de Datos
def build_binary_dataset(output_root=DATA_DIR):
    # Lógica de guarda: Si la carpeta train existe, no descargamos de nuevo
    if output_root.exists() and (output_root / "train").exists():
        print(f"El dataset ya existe en {output_root.name}. Saltando descarga.")
        return count_images(output_root)

    print("Preparando dataset...")
    raw_download_path = Path(kagglehub.dataset_download(DATASET_SLUG))
    dataset_root = find_dataset_root(raw_download_path)
    grouped_paths = collect_images(dataset_root)
    splits = stratified_split(grouped_paths, seed=SEED)

    if output_root.exists():
        shutil.rmtree(output_root)

    for split_name in ["train", "val", "test"]:
        for label in ["Healthy", "Rotten"]:
            (output_root / split_name / label).mkdir(parents=True, exist_ok=True)

    jobs = []
    for split_name, split_labels in splits.items():
        for label, paths in split_labels.items():
            out_dir = output_root / split_name / label
            for index, src in enumerate(paths):
                dst = out_dir / f"{label.lower()}_{index:07d}{src.suffix.lower()}"
                jobs.append((src, dst))

    with ThreadPoolExecutor(max_workers=min(16, os.cpu_count() or 8)) as executor:
        list(tqdm(executor.map(lambda item: link_or_copy(*item), jobs), total=len(jobs), desc="Preparando dataset"))

    return count_images(output_root)

def choose_sample_image(test_root=DATA_DIR / "test"):
    for label in ["Healthy", "Rotten"]:
        label_dir = test_root / label
        if not label_dir.exists():
            continue
        for path in sorted(label_dir.glob("*")):
            if path.suffix.lower() in IMAGE_EXTS:
                return path
    raise FileNotFoundError("No se encontro una imagen de prueba en DATA_DIR/test")

def metrics_to_dataframe(split_name, metrics):
    rows = []
    results_dict = getattr(metrics, "results_dict", {}) or {}
    for key, value in results_dict.items():
        if isinstance(value, (int, float)):
            rows.append({
                "split": split_name,
                "metric": key,
                "value": float(value),
            })
    return pd.DataFrame(rows)

# 6. Ejecutamos la validación/creación del dataset
resumen_dataset = build_binary_dataset()

print("\n--- Resumen de Imágenes ---")
for split, labels in resumen_dataset.items():
    print(f"[{split.upper()}] Sanas: {labels.get('Healthy', 0)} | Podridas: {labels.get('Rotten', 0)}")

El dataset ya existe en fruit_binary_yolo_cls. Saltando descarga.

--- Resumen de Imágenes ---
[TRAIN] Sanas: 11029 | Podridas: 12403
[VAL] Sanas: 1378 | Podridas: 1550
[TEST] Sanas: 1380 | Podridas: 1551


In [4]:
dataset_summary = build_binary_dataset()
dataset_summary

El dataset ya existe en fruit_binary_yolo_cls. Saltando descarga.


{'train': {'Healthy': 11029, 'Rotten': 12403},
 'val': {'Healthy': 1378, 'Rotten': 1550},
 'test': {'Healthy': 1380, 'Rotten': 1551}}

In [5]:


# 1. DEFINICIÓN DE RUTAS (Ruta absoluta de Lustre)
# Usamos la que me confirmaste donde estás trabajando
RUTA_PROYECTO = Path("/lustre/home/estudiante_66/computerVision/computerVision")
RUTA_MLRUNS = RUTA_PROYECTO / "mlruns"

# Configuración de MLflow
mlflow.set_tracking_uri(f"file://{RUTA_MLRUNS}")
mlflow.set_experiment("Clasificacion_Frutas_Sanas_vs_Podridas")

print(f" Proyecto en: {RUTA_PROYECTO}")
print(f" Registros en: {RUTA_MLRUNS}")

# 2. INICIO DEL ENTRENAMIENTO
# Usamos el bloque 'with' para asegurar que la corrida se cierre correctamente
with mlflow.start_run(run_name=TRAIN_RUN_NAME) as run:
    
    # Cargamos el modelo base
    model = YOLO(BASE_MODEL)
    
    # Entrenamiento
    # Nota: 'project' ahora apunta a tu carpeta de Lustre directamente
    model.train(
        data=str(DATA_DIR),
        task="classify",
        imgsz=640,
        epochs=5,
        batch=32,
        device=DEVICE,
        project=str(RUTA_PROYECTO / "runs"), # Se guardará en /computerVision/runs
        name=TRAIN_RUN_NAME,
        pretrained=True,
        deterministic=True,
        verbose=True,
        workers=0,
        exist_ok=True,
        freeze=10
    )

    # 3. VALIDACIÓN Y GUARDADO DE ARTEFACTOS
    # En lugar de armar la ruta a mano, se la pedimos a YOLO
    if hasattr(model, 'trainer') and model.trainer is not None:
        directorio_entrenamiento = Path(model.trainer.save_dir)
        best_weights = directorio_entrenamiento / "weights" / "best.pt"
    else:
        # Respaldo manual si el objeto no tiene el atributo
        best_weights = RUTA_PROYECTO / "runs" / TRAIN_RUN_NAME / "weights" / "best.pt"

    # Verificación de seguridad
    if best_weights.exists():
        print(f" Pesos localizados en: {best_weights}")
        # Subimos el mejor modelo como artefacto a MLflow
        mlflow.log_artifact(str(best_weights), artifact_path="modelo_final_yolo")
        print(" ¡Modelo y métricas guardados exitosamente en MLflow!")
    else:
        print(f" Error: No se encontró best.pt en {best_weights}")
        # Listamos contenido para debug si llega a fallar
        print("Contenido de la carpeta de entrenamiento:")
        !ls -R {str(RUTA_PROYECTO / "runs")}

 Proyecto en: /lustre/home/estudiante_66/computerVision/computerVision
 Registros en: /lustre/home/estudiante_66/computerVision/computerVision/mlruns
Ultralytics 8.4.24 🚀 Python-3.9.21 torch-2.3.1+rocm5.7 CUDA:0 (AMD Instinct MI210, 65520MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/lustre/cursos/curso04/estudiante_66/computerVision/fruit_binary_yolo_cls, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.

2026/03/25 17:51:21 INFO mlflow.tracking.fluent: Experiment with name '/lustre/home/estudiante_66/computerVision/computerVision/runs' does not exist. Creating a new experiment.


MLflow: logging run_id(95aa7a1c92bc412ab881c98ef4fe7d68) to file:///lustre/home/estudiante_66/computerVision/computerVision/mlruns
MLflow: disable with 'yolo settings mlflow=False'
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to /lustre/cursos/curso04/estudiante_66/computerVision/runs/fruit_hs_vs_rt_cls_simple_ft5
Starting training for 5 epochs...

      Epoch    GPU_mem       loss  Instances       Size
        1/5     0.104G     0.2946          8        640: 100% ━━━━━━━━━━━━ 733/733 1.0it/s 11:50<0.5s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 46/46 1.4s/it 1:061.5sss
                   all       0.96          1

      Epoch    GPU_mem       loss  Instances       Size
        2/5     0.787G     0.1651          8        640: 100% ━━━━━━━━━━━━ 733/733 1.5it/s 8:13<0.4s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 46/46 1.2it/s 39.9s0.9ss
                   all      0.973          1

      Epoch    GPU_mem       loss  

In [8]:
!find runs -name 'best.pt'

runs/classify/runs/classify/fruit_hs_vs_rt_cls_simple_ft5/weights/best.pt


In [18]:

client = mlflow.tracking.MlflowClient()

# Esto nos dirá qué experimentos reconoce el sistema REALMENTE
for rm in client.search_experiments():
    print(f"ID: {rm.experiment_id} | Nombre: {rm.name} | Ruta: {rm.artifact_location}")

ID: 920878481846248033 | Nombre: runs/classify | Ruta: file:///lustre/cursos/curso04/estudiante_66/computerVision/mlruns/920878481846248033
ID: 727963899540270571 | Nombre: Clasificacion_Frutas_Sanas_vs_Podridas | Ruta: file:///lustre/cursos/curso04/estudiante_66/computerVision/mlruns/727963899540270571
ID: 0 | Nombre: Default | Ruta: file:///lustre/cursos/curso04/estudiante_66/computerVision/mlruns/0


In [8]:
# 1. Buscamos al "cerebro" más inteligente que resultó del entrenamiento
if "best_weights" not in globals():
    best_weights = Path("runs/classify") / TRAIN_RUN_NAME / "weights" / "best.pt"

classifier_model = YOLO(str(best_weights))

# Corremos las evaluaciones (YOLO mandará un resumen a MLflow automáticamente)
print("\nEvaluando set de Validación...")
val_metrics = classifier_model.val(data=str(DATA_DIR), split="val", imgsz=640, device=DEVICE)
print("\n Evaluando el set de test..")
test_metrics = classifier_model.val(data=str(DATA_DIR), split="test", imgsz=640, device=DEVICE)

# se arma el dataf
metrics_df = pd.concat([
    metrics_to_dataframe("val", val_metrics),
    metrics_to_dataframe("test", test_metrics),
], ignore_index=True)

# Mostramos la tabla
print("\n Resultados Finales:")
display(metrics_df)


Evaluando set de Validación...
Ultralytics 8.4.24 🚀 Python-3.9.21 torch-2.3.1+rocm5.7 CUDA:0 (AMD Instinct MI210, 65520MiB)
YOLO11n-cls summary (fused): 47 layers, 1,528,586 parameters, 0 gradients, 3.2 GFLOPs
train: /lustre/cursos/curso04/estudiante_66/computerVision/fruit_binary_yolo_cls/train... found 23432 images in 2 classes ✅ 
val: /lustre/cursos/curso04/estudiante_66/computerVision/fruit_binary_yolo_cls/val... found 2928 images in 2 classes ✅ 
test: /lustre/cursos/curso04/estudiante_66/computerVision/fruit_binary_yolo_cls/test... found 2931 images in 2 classes ✅ 
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 499.1±785.1 MB/s, size: 39.7 KB)
val: Scanning /lustre/cursos/curso04/estudiante_66/computerVision/fruit_binary_yolo_cls/val... 2928 images, 0 corrupt: 100% ━━━━━━━━━━━━ 2928/2928 614.0Mit/s 0.0s
val: /lustre/cursos/curso04/estudiante_66/computerVision/fruit_binary_yolo_cls/val/Healthy/healthy_0000754.jpeg: corrupt JPEG restored and saved
val: /lustre/cursos/curso04/est

/lustre/cursos/curso04/estudiante_66/computerVision/.venv/lib64/python3.9/site-packages/torch/utils/data/dataloader.py:558: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 183/183 13.1it/s 14.0s0.2s
                   all      0.987          1
Speed: 0.3ms preprocess, 0.9ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to /lustre/cursos/curso04/estudiante_66/computerVision/runs/classify/val

 Evaluando el set de test..
Ultralytics 8.4.24 🚀 Python-3.9.21 torch-2.3.1+rocm5.7 CUDA:0 (AMD Instinct MI210, 65520MiB)
train: /lustre/cursos/curso04/estudiante_66/computerVision/fruit_binary_yolo_cls/train... found 23432 images in 2 classes ✅ 
val: /lustre/cursos/curso04/estudiante_66/computerVision/fruit_binary_yolo_cls/val... found 2928 images in 2 classes ✅ 
test: /lustre/cursos/curso04/estudiante_66/computerVision/fruit_binary_yolo_cls/test... found 2931 images in 2 classes ✅ 
test: Fast image access ✅ (ping: 0.5±0.2 ms, read: 9.2±7.8 MB/s, size: 98.2 KB)
test: Scanning /lustre/cursos/curso04/estudiante_66/computerVision/fruit_binary_yolo_cls/test... 2931 images, 0 corrupt: 100% 

/lustre/cursos/curso04/estudiante_66/computerVision/.venv/lib64/python3.9/site-packages/torch/utils/data/dataloader.py:558: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 184/184 11.1it/s 16.6s0.2s
                   all       0.99          1
Speed: 0.3ms preprocess, 1.0ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to /lustre/cursos/curso04/estudiante_66/computerVision/runs/classify/val2

 Resultados Finales:


,split,metric,value
0,val,metrics/accuracy_top1,0.986680
1,val,metrics/accuracy_top5,1.000000
2,val,fitness,0.993340
3,test,metrics/accuracy_top1,0.990447
4,test,metrics/accuracy_top5,1.000000
5,test,fitness,0.995223


In [ ]:
if metrics_df.empty:
    print("No se encontraron metricas numericas para graficar.")
else:
    pivot_df = metrics_df.pivot(index="metric", columns="split", values="value")
    ax = pivot_df.plot(kind="bar", figsize=(12, 6))
    ax.set_title("Metricas de validacion y test")
    ax.set_ylabel("Valor")
    ax.set_xlabel("Metrica")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

In [ ]:
detector = YOLO(DETECTOR_WEIGHTS)
classifier = YOLO(str(best_weights))

def clamp_box(x1, y1, x2, y2, width, height):
    x1 = max(0, min(int(x1), width - 1))
    y1 = max(0, min(int(y1), height - 1))
    x2 = max(0, min(int(x2), width - 1))
    y2 = max(0, min(int(y2), height - 1))
    if x2 <= x1:
        x2 = min(width - 1, x1 + 1)
    if y2 <= y1:
        y2 = min(height - 1, y1 + 1)
    return x1, y1, x2, y2

def detect_and_count(image_path, det_conf=0.20, det_iou=0.60, min_box_area=32 * 32, only_produce=True):
    image_path = Path(image_path)
    bgr = cv2.imread(str(image_path))
    if bgr is None:
        raise FileNotFoundError(f"No se pudo leer imagen: {image_path}")

    height, width = bgr.shape[:2]
    det_result = detector.predict(
        source=bgr,
        conf=det_conf,
        iou=det_iou,
        device=DEVICE,
        verbose=False,
    )[0]

    instances = []
    healthy_count = 0
    rotten_count = 0
    names = det_result.names

    for box in det_result.boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        x1, y1, x2, y2 = clamp_box(x1, y1, x2, y2, width, height)

        area = (x2 - x1) * (y2 - y1)
        if area < min_box_area:
            continue

        class_index = int(box.cls[0].item())
        det_name = names[class_index]
        det_score = float(box.conf[0].item())

        if only_produce and PRODUCE_CLASS_NAMES is not None and det_name not in PRODUCE_CLASS_NAMES:
            continue

        crop = bgr[y1:y2, x1:x2]
        if crop.size == 0:
            continue

        cls_result = classifier.predict(source=crop, imgsz=640, device=DEVICE, verbose=False)[0]
        top_index = int(cls_result.probs.top1)
        health_label = cls_result.names[top_index]
        health_conf = float(cls_result.probs.top1conf.item())

        label_lower = health_label.lower()
        if "healthy" in label_lower:
            healthy_count += 1
            box_color = (60, 180, 75)
        elif "rotten" in label_lower:
            rotten_count += 1
            box_color = (230, 25, 75)
        else:
            box_color = (255, 165, 0)

        text = f"{health_label} {health_conf:.2f} | {det_name} {det_score:.2f}"
        cv2.rectangle(bgr, (x1, y1), (x2, y2), box_color, 2)
        cv2.putText(bgr, text, (x1, max(20, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, box_color, 2, cv2.LINE_AA)

        instances.append({
            "bbox_xyxy": (x1, y1, x2, y2),
            "det_class": det_name,
            "det_conf": det_score,
            "health_label": health_label,
            "health_conf": health_conf,
        })

    summary = {
        "healthy": healthy_count,
        "rotten": rotten_count,
        "total": healthy_count + rotten_count,
    }

    return bgr, instances, summary

In [ ]:
IMAGE_PATH = choose_sample_image()
OUTPUT_IMAGE_PATH = Path("sample_result.jpg")

print("Imagen de prueba:", IMAGE_PATH)

annotated_bgr, instances, summary = detect_and_count(
    IMAGE_PATH,
    det_conf=0.20,
    det_iou=0.60,
    min_box_area=32 * 32,
    only_produce=True,
)

cv2.imwrite(str(OUTPUT_IMAGE_PATH), annotated_bgr)

print(f"Conteo final -> {summary['rotten']} podridas, {summary['healthy']} sanas")
print(f"Instancias analizadas: {summary['total']}")
print("Imagen guardada en:", OUTPUT_IMAGE_PATH)

rgb = cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(12, 8))
plt.imshow(rgb)
plt.axis("off")
plt.title(f"Resultado: {summary['rotten']} Rotten, {summary['healthy']} Healthy")
plt.show()

pd.DataFrame(instances)

In [ ]:
INPUT_DIR = Path("inference_images")
OUTPUT_DIR = Path("inference_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

rows = []
for image_path in sorted(INPUT_DIR.glob("*")):
    if image_path.suffix.lower() not in IMAGE_EXTS:
        continue

    annotated, instances, summary = detect_and_count(image_path, only_produce=True)
    out_image = OUTPUT_DIR / image_path.name
    cv2.imwrite(str(out_image), annotated)

    rows.append({
        "image": image_path.name,
        "healthy": summary["healthy"],
        "rotten": summary["rotten"],
        "total": summary["total"],
    })

df = pd.DataFrame(rows)
csv_path = OUTPUT_DIR / "counts.csv"
df.to_csv(csv_path, index=False)

print("CSV generado en:", csv_path)
df.head()

# 

## Refactor: Evaluación + MLflow con pesos ya entrenados

Bloque modular pensado para ejecutarse **sin reentrenar**. Sólo edita las rutas en la celda de configuración para alternar entre Windows y la ruta de Lustre en el clúster.

**Flujo:**
1. Configuración de rutas, hiperparámetros y validación de artefactos.
2. Carga de `best.pt` y evaluación explícita sobre TEST (Top-1, Top-5, matriz de confusión, gráficas).
3. Registro robusto en MLflow (parámetros, métricas y artefactos `.pt` + plots).
4. Inferencia demostrativa sobre 5 imágenes aleatorias del test.

**Nota sobre gráficas:** para tareas `classify`, YOLO genera únicamente `confusion_matrix.png` y `confusion_matrix_normalized.png`. Las curvas F1 / PR son propias de tareas de detección y no se generarán en este flujo; el código las detecta opcionalmente si existieran.

In [ ]:

# --- RUTAS EDITABLES (cambia aqui cuando pases Windows <-> Lustre) --------
# Windows local : r"C:\Users\joel_\OneDrive\Documentos\computerVision\runs\classify\fruit_hs_vs_rt_cls_simple_ft5\weights\best.pt"
# Lustre clúster: "/lustre/cursos/curso04/estudiante_66/computerVision/runs/classify/fruit_hs_vs_rt_cls_simple_ft5/weights/best.pt"
PATH_BEST_WEIGHTS = Path("runs/classify/runs/classify/fruit_hs_vs_rt_cls_simple_ft5/weights/best.pt")
PATH_DATA         = Path(DATA_DIR)      # reutiliza DATA_DIR definido arriba
PATH_MLRUNS       = Path("mlruns")      # local; en clúster: "/lustre/cursos/curso04/estudiante_66/computerVision/mlruns"
MLFLOW_EXPERIMENT = "Clasificacion_Frutas_Sanas_vs_Podridas"

# --- HIPERPARAMETROS DE INFERENCIA (trazabilidad en MLflow) ---------------
EVAL_IMGSZ = 640
EVAL_BATCH = 32
EVAL_CONF  = 0.25   # solo se registra; en classify YOLO no lo usa
EVAL_IOU   = 0.60

# --- VALIDACION DE ARCHIVOS ANTES DE CONTINUAR ----------------------------
def _build_tracking_uri(path: Path) -> str:
    """Genera un URI file:// multiplataforma (Windows y Linux)."""
    path = path.resolve()
    path.mkdir(parents=True, exist_ok=True)
    return path.as_uri()  # en Windows -> file:///C:/..., en Linux -> file:///lustre/...

if not PATH_BEST_WEIGHTS.exists():
    raise FileNotFoundError(
        f"No se encontro best.pt en: {PATH_BEST_WEIGHTS}\n"
        "Edita PATH_BEST_WEIGHTS con la ruta correcta (Windows o Lustre)."
    )
if not PATH_DATA.exists():
    raise FileNotFoundError(f"No se encontro el dataset en: {PATH_DATA}")

MLFLOW_TRACKING_URI = _build_tracking_uri(PATH_MLRUNS)
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)

# Desactiva el autolog de Ultralytics para que no cree runs paralelos
# al ejecutar model.val(); si lo quieres activo ponlo en True.
try:
    from ultralytics import settings as _ul_settings
    _ul_settings.update({"mlflow": False})
except Exception:
    pass

print(f"Pesos      : {PATH_BEST_WEIGHTS}")
print(f"Dataset    : {PATH_DATA}")
print(f"MLflow URI : {MLFLOW_TRACKING_URI}")
print(f"Experimento: {MLFLOW_EXPERIMENT}")

# --- CARGA DEL MODELO DESDE LOS PESOS -------------------------------------
model = YOLO(str(PATH_BEST_WEIGHTS))
CLASS_NAMES = list(model.names.values()) if isinstance(model.names, dict) else list(model.names)
print(f"Modelo cargado | clases: {CLASS_NAMES}")


In [ ]:

# Evaluacion sobre TEST con extraccion explicita de metricas y artefactos.
import numpy as np

test_metrics = model.val(
    data=str(PATH_DATA),
    split="test",
    imgsz=EVAL_IMGSZ,
    batch=EVAL_BATCH,
    device=DEVICE,
    plots=True,
    verbose=False,
)

# --- Metricas escalares ----------------------------------------------------
results_dict = getattr(test_metrics, "results_dict", {}) or {}
top1 = float(results_dict.get("metrics/accuracy_top1", float("nan")))
top5 = float(results_dict.get("metrics/accuracy_top5", float("nan")))
fitness = float(results_dict.get("fitness", float("nan")))

# --- Matriz de confusion ---------------------------------------------------
cm_matrix = None
if getattr(test_metrics, "confusion_matrix", None) is not None:
    cm_matrix = np.asarray(test_metrics.confusion_matrix.matrix)

# --- Graficas generadas por YOLO en save_dir -------------------------------
# Nota: para task="classify" YOLO NO genera F1_curve/PR_curve;
# solo confusion_matrix(_normalized). Se contemplan por compatibilidad.
eval_save_dir = Path(test_metrics.save_dir) if hasattr(test_metrics, "save_dir") else None
plot_files: dict[str, Path] = {}
if eval_save_dir and eval_save_dir.exists():
    candidatos = [
        "confusion_matrix.png",
        "confusion_matrix_normalized.png",
        "F1_curve.png",
        "PR_curve.png",
        "P_curve.png",
        "R_curve.png",
        "results.png",
    ]
    for nombre in candidatos:
        ruta = eval_save_dir / nombre
        if ruta.exists():
            plot_files[nombre] = ruta

metrics_summary = pd.DataFrame([
    {"metric": "accuracy_top1", "value": top1},
    {"metric": "accuracy_top5", "value": top5},
    {"metric": "fitness",       "value": fitness},
])

print("--- Metricas en TEST ---")
print(metrics_summary.to_string(index=False))
print(f"\nMatriz de confusion:\n{cm_matrix}")
print(f"\nArtefactos generados en: {eval_save_dir}")
for nombre, ruta in plot_files.items():
    print(f"  - {nombre} -> {ruta}")


In [ ]:

# Bloque robusto de MLflow: registra parametros, metricas y artefactos
# en una corrida independiente. Cada ejecucion crea un nuevo run_name.
import tempfile

def log_evaluation_to_mlflow(
    run_name: str = "eval_test_best_pt",
    extra_params: dict | None = None,
) -> str:
    """Registra la evaluacion actual en MLflow y retorna el run_id generado.

    Parametros:
      run_name     : nombre visible de la corrida.
      extra_params : dict opcional con parametros adicionales a registrar.
    """
    params = {
        "weights_path": str(PATH_BEST_WEIGHTS),
        "data_path":    str(PATH_DATA),
        "imgsz":        EVAL_IMGSZ,
        "batch":        EVAL_BATCH,
        "conf":         EVAL_CONF,
        "iou":          EVAL_IOU,
        "device":       str(DEVICE),
        "split":        "test",
        "num_classes":  len(CLASS_NAMES),
        "task":         "classify",
    }
    if extra_params:
        params.update(extra_params)

    metrics = {
        "accuracy_top1": top1,
        "accuracy_top5": top5,
        "fitness":       fitness,
    }

    with mlflow.start_run(run_name=run_name) as run:
        mlflow.log_params(params)
        # Filtramos NaN: MLflow los rechaza en log_metric.
        mlflow.log_metrics({k: v for k, v in metrics.items() if not np.isnan(v)})
        mlflow.set_tag("classes", ",".join(CLASS_NAMES))

        # Artefacto principal: el .pt.
        mlflow.log_artifact(str(PATH_BEST_WEIGHTS), artifact_path="modelo")

        # Graficas generadas por YOLO en save_dir.
        for name, path in plot_files.items():
            try:
                mlflow.log_artifact(str(path), artifact_path="plots")
            except Exception as exc:
                print(f"  [WARN] No se pudo subir {name}: {exc}")

        # Matriz de confusion tambien como CSV (util para analisis posterior).
        if cm_matrix is not None:
            cm_df = pd.DataFrame(
                cm_matrix,
                index=[f"true_{c}" for c in CLASS_NAMES],
                columns=[f"pred_{c}" for c in CLASS_NAMES],
            )
            with tempfile.TemporaryDirectory() as tmp:
                cm_csv = Path(tmp) / "confusion_matrix.csv"
                cm_df.to_csv(cm_csv)
                mlflow.log_artifact(str(cm_csv), artifact_path="plots")

        # Tabla de metricas como CSV.
        with tempfile.TemporaryDirectory() as tmp:
            metrics_csv = Path(tmp) / "metrics_test.csv"
            metrics_summary.to_csv(metrics_csv, index=False)
            mlflow.log_artifact(str(metrics_csv), artifact_path="metrics")

        print(f"MLflow run creado | run_id = {run.info.run_id}")
        return run.info.run_id

run_id = log_evaluation_to_mlflow()
print(f"URI activo: {mlflow.get_tracking_uri()}")


In [ ]:

# Inferencia sobre 5 imagenes aleatorias del split de test.
# Uso torch.inference_mode() y liberacion explicita para minimizar memoria.
import gc
import random as _random

NUM_SAMPLES = 5
TEST_DIR = PATH_DATA / "test"

if not TEST_DIR.exists():
    raise FileNotFoundError(f"No existe la carpeta de test: {TEST_DIR}")

all_test_images = [
    p for p in TEST_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
]
if len(all_test_images) < NUM_SAMPLES:
    raise RuntimeError(
        f"Se requieren al menos {NUM_SAMPLES} imagenes en test; hay {len(all_test_images)}."
    )

sample_paths = _random.Random(42).sample(all_test_images, NUM_SAMPLES)

fig, axes = plt.subplots(1, NUM_SAMPLES, figsize=(4 * NUM_SAMPLES, 4))
if NUM_SAMPLES == 1:
    axes = [axes]

with torch.inference_mode():
    for ax, img_path in zip(axes, sample_paths):
        # Una imagen a la vez -> evita cargar un batch completo en memoria.
        result = model.predict(
            source=str(img_path),
            imgsz=EVAL_IMGSZ,
            device=DEVICE,
            verbose=False,
        )[0]

        top_idx  = int(result.probs.top1)
        pred     = result.names[top_idx]
        conf     = float(result.probs.top1conf.item())
        true_lbl = img_path.parent.name

        with Image.open(img_path) as im:
            ax.imshow(im.convert("RGB"))
        ax.set_title(f"GT: {true_lbl}\nPred: {pred} ({conf:.2f})", fontsize=10)
        ax.axis("off")

        # Libera el objeto de resultado antes de pasar a la siguiente imagen.
        del result

plt.tight_layout()
plt.show()

# Limpieza final para no dejar tensores colgados en GPU.
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
